# Nettoyage Avancé — Dataset Commentaires

Applique des filtres agressifs sur tout fichier CSV avec une colonne `text` :
- Gibberish / keyboard mashing
- Textes sans contenu réel (ni arabe ni latin)
- Répétition excessive d'un caractère ou d'un mot
- Séquences aléatoires longues sans voyelle

**Paramètres :** modifier `INPUT_CSV` et `OUTPUT_CSV` dans la cellule suivante.

In [1]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

# ── PARAMETRES (modifier ici pour chaque client) ──────────────────
INPUT_CSV  = 'DATA TOPNET/topnet_all.csv'
OUTPUT_CSV = 'DATA TOPNET/topnet_all_clean.csv'
# ──────────────────────────────────────────────────────────────────

# Dossier outputs pour les graphiques
os.makedirs('outputs', exist_ok=True)

df = pd.read_csv(INPUT_CSV, encoding='utf-8')

# Normaliser: on garde uniquement la colonne text
if 'text' in df.columns:
    df = df[['text']].rename(columns={'text': 'comment'})
elif 'comment' in df.columns:
    df = df[['comment']]
else:
    col = df.select_dtypes(include='object').columns[-1]
    df = df[[col]].rename(columns={col: 'comment'})

df['comment'] = df['comment'].astype(str).str.strip()

print(f'Charge     : {len(df):,} commentaires')
print(f'Moy. chars : {df["comment"].str.len().mean():.0f}')


Charge     : 2,596 commentaires
Moy. chars : 44


## 1. Définition des filtres

In [2]:
ARABIC_RE  = re.compile(r'[؀-ۿ]')
LATIN_RE   = re.compile(r'[a-zA-Z]')
EMOJI_RE   = re.compile(
    u'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
    u'\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF'
    u'\U00002700-\U000027BF\U0001F900-\U0001F9FF'
    u'\U00002600-\U000026FF❤♥♦]+', flags=re.UNICODE
)


def has_gibberish_latin(text):
    """Séquence de 7+ consonnes latines consécutives = keyboard mashing."""
    return bool(re.search(r'(?i)[bcdfghjklmnpqrstvwxyz]{7,}', text))


def has_high_char_repeat(text, threshold=0.28):
    """Un seul caractère alphabétique représente >28% du texte total."""
    t = re.sub(r'\s', '', text.lower())
    if len(t) == 0:
        return True
    for ch in set(t):
        if ch.isalpha() and t.count(ch) / len(t) > threshold:
            return True
    return False


def has_repetitive_words(text, threshold=0.45):
    """Même mot dépasse 45% de tous les mots (ex: 'أنا أنا أنا أنا')."""
    words = text.lower().split()
    if len(words) < 5:
        return False
    from collections import Counter
    freq = Counter(words)
    return freq.most_common(1)[0][1] / len(words) > threshold


def has_no_real_content(text):
    """Ni arabe ni latin réel (que chiffres, emojis, ponctuation)."""
    return not ARABIC_RE.search(text) and not LATIN_RE.search(text)


def is_gibberish_random(text):
    """
    Détecte les séquences aléatoires de caractères (type mot de passe ou spam).
    Critère : mot de 15+ chars sans voyelle arabe ni voyelle latine.
    """
    words = text.split()
    for w in words:
        w_clean = re.sub(r'[^a-zA-Z؀-ۿ]', '', w)
        if len(w_clean) >= 15:
            vowels_lat = len(re.findall(r'[aeiouAEIOU]', w_clean))
            vowels_ar  = len(re.findall(r'[َ-ِاوي]', w_clean))
            if vowels_lat + vowels_ar == 0:
                return True
    return False


# ── Filtre principal ──────────────────────────────────────────────────────────
def should_keep(text):
    """Retourne (True/False, raison)."""
    text = str(text).strip()

    # Vide ou nan
    if not text or text == 'nan':
        return False, 'vide'

    # Aucun contenu réel (ni arabe ni latin)
    if has_no_real_content(text):
        return False, 'pas_de_contenu'

    # Keyboard mashing Latin
    if has_gibberish_latin(text):
        return False, 'gibberish_latin'

    # Mot aléatoire très long sans voyelle
    if is_gibberish_random(text):
        return False, 'gibberish_random'

    # Répétition excessive d'un seul caractère
    if has_high_char_repeat(text):
        return False, 'char_repeat'

    # Répétition excessive d'un seul mot
    if has_repetitive_words(text):
        return False, 'word_repeat'

    return True, 'ok'


print('Filtres définis (sans filtre de longueur).')

Filtres définis (sans filtre de longueur).


## 2. Application des filtres

In [3]:
results = df['comment'].apply(should_keep)
df['keep']   = results.apply(lambda x: x[0])
df['reason'] = results.apply(lambda x: x[1])

df_clean    = df[df['keep']].copy()
df_rejected = df[~df['keep']].copy()

print(f'Total       : {len(df):,}')
print(f'Conservés   : {len(df_clean):,}  ({len(df_clean)/len(df)*100:.1f}%)')
print(f'Supprimés   : {len(df_rejected):,}  ({len(df_rejected)/len(df)*100:.1f}%)')
print(f'\n── Raisons de suppression ──')
print(df_rejected['reason'].value_counts().to_string())

Total       : 2,596
Conservés   : 2,581  (99.4%)
Supprimés   : 15  (0.6%)

── Raisons de suppression ──
reason
char_repeat       13
pas_de_contenu     2


## 3. Aperçu de ce qui est supprimé (vérification)

In [4]:
for reason in df_rejected['reason'].value_counts().index:
    subset = df_rejected[df_rejected['reason'] == reason].head(5)
    print(f'\n── {reason} ({len(df_rejected[df_rejected["reason"]==reason])} total) ──')
    for _, row in subset.iterrows():
        print(f'  | {row["comment"][:90]}')


── char_repeat (13 total) ──
  | Brabbi. ?
  | Brabbi. ??
  | Reseau tayara barcha ❤️ 🔥
  | reseau tayara barcha ❤️
  | Ghalia barcha

── pas_de_contenu (2 total) ──
  | ?? ???? ???? ?? ??????... ???? ??????
  | ??? ??????? *101# ????? ??? ???????


## 4. Aperçu de ce qui est conservé

In [5]:
print(f'── Échantillon aléatoire de {min(30, len(df_clean))} commentaires conservés ──\n')
sample = df_clean.sample(min(30, len(df_clean)), random_state=42)
for _, row in sample.iterrows():
    print(f'  | {row["comment"][:100]}')

── Échantillon aléatoire de 30 commentaires conservés ──

  | Merci. ?
  | internet waqfet waqt امتحانات في جامعة صفاقس 😤
  | Merci 3ala l offre mla7. ??
  | ???? ?? ?????? ??????? code par SMS b sarii3a?
  | N7eb nbadal l'offre chna3mel? 🤔
  | bech n9addi el lila ta3 ras el 3am online w connexion bata2 🥲
  | portee du wifi limitee, puis-je ajouter un repeteur?
  | service behya barcha 👌 réseau stable fi Sfax nchallah يتحسن yaatikom essa7a 😅 ena client depuis snin
  | Appli تحتاج option e-SIM activation 📱
  | el teknicien ji w 9al mochkla fi el bra, ba3da ma ja 7ata
  | chkoun y3awedni bil facture? zedou 3liya w ma3reftech leh
  | Winhom les points fidelité? 📶
  | service client ما يجاوبش nchallah يتحسن 👍 mais bon service محترم
  | 4G في Oued Sbaihia باهي برشا 😊
  | comment bloquer certains sites sur le routeur pour les enfants?
  | Merci Topnet, n3ayetkom barcha w t5admouni mli7.
  | Code promo n7eb n3abbi ra9mi, win najm nal9ah?
  | كيفاش نربط طابعة لاسلكية بالراوتر تاع توبنت؟
  | se

## 5. Ajustement du seuil de longueur minimale (optionnel)

Si tu veux être plus ou moins strict, modifie `MIN_LEN` et `MIN_WORDS` dans la cellule ci-dessous
et relance depuis la cellule 5.

In [6]:
# Distribution des longueurs des commentaires conservés
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
lengths = df_clean['comment'].str.len()
ax.hist(lengths, bins=50, color='steelblue', edgecolor='white')
ax.set(title='Distribution des longueurs (chars)', xlabel='Longueur', ylabel='Count')
ax.axvline(15, color='red', linestyle='--', label='Seuil min (15)')
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
reasons_counts = df_rejected['reason'].value_counts()
ax2.barh(reasons_counts.index, reasons_counts.values, color='salmon', edgecolor='white')
ax2.set(title='Raisons de suppression', xlabel='Count')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('outputs/nettoyage_avance_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé → outputs/nettoyage_avance_stats.png')

Sauvegardé → outputs/nettoyage_avance_stats.png


C:\Users\emnam\AppData\Local\Temp\ipykernel_28208\2732187886.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Sauvegarder le dataset nettoyé

**Vérifie les cellules 3 et 4 avant de continuer.**  
Si des commentaires valides sont supprimés à tort, ajuste les seuils dans `should_keep()`.

In [7]:
import os
os.makedirs('../outputs', exist_ok=True)

df_out = df_clean[['comment']].reset_index(drop=True)
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Dataset nettoyé sauvegardé → {OUTPUT_CSV}')
print(f'  {len(df):,} → {len(df_out):,} commentaires  ({len(df_out)/len(df)*100:.1f}% conservés)')
print(f'\nProchaine étape : relancer pseudo_labeling.ipynb')
print(f'  → Mettre UNLABELED = "../data/processed/dataset_commentaires_super_clean.csv"')

Dataset nettoyé sauvegardé → DATA TOPNET/topnet_all_clean.csv
  2,596 → 2,581 commentaires  (99.4% conservés)

Prochaine étape : relancer pseudo_labeling.ipynb
  → Mettre UNLABELED = "../data/processed/dataset_commentaires_super_clean.csv"
